# Catan research playground

Research notebook for exploring `catanatron` and the bots in this repo.

Every reusable piece of logic lives in [../src/catan_bots](../src/catan_bots)
and is covered by tests in [../tests](../tests). This notebook only *calls*
those modules — it never redefines them. When something prototyped here proves
useful, promote it into a module and add a test.

Run `uv sync` first — that installs `catan_bots` into `.venv` — then open
this notebook with the project's `.venv` kernel.

## 1. Setup

In [1]:
from catanatron import Color
from catanatron.models.player import Player, RandomPlayer
from catanatron.models.enums import ActionType
from catanatron.players.search import VictoryPointPlayer
from catanatron.players.weighted_random import WeightedRandomPlayer

from catan_bots import (
    ActionTypeCounter,
    BoardInspector,
    GameFactory,
    GameReport,
    GreedySettlerBot,
    Tournament,
)

print("Imports OK")

Imports OK


## 2. Building reproducible games

`GameFactory` ([src/catan_bots/games.py](../src/catan_bots/games.py)) owns game
construction: it resets every player between games and makes any integer seed
reproducible, including `0` — which plain `catanatron.Game` silently treats as
"no seed".

In [2]:
factory = GameFactory(
    [
        RandomPlayer(Color.RED),
        RandomPlayer(Color.BLUE),
        RandomPlayer(Color.ORANGE),
        RandomPlayer(Color.WHITE),
    ]
)
game = factory.create(seed=42)

print(factory)
print(f"Game ID  : {game.id}")
print(f"VP target: {game.vps_to_win}")
print(f"Seating  : {game.state.colors}")

GameFactory([RandomPlayer:RED, RandomPlayer:BLUE, RandomPlayer:ORANGE, RandomPlayer:WHITE], vps_to_win=10)
Game ID  : 6f083231-2680-4172-b3c0-274ae6424a45
VP target: 10
Seating  : (<Color.RED: 'RED'>, <Color.WHITE: 'WHITE'>, <Color.BLUE: 'BLUE'>, <Color.ORANGE: 'ORANGE'>)


## 3. Reading game state

`GameReport` ([src/catan_bots/analytics/reports.py](../src/catan_bots/analytics/reports.py))
snapshots the board, the players, and the action log. It reads state eagerly,
so a report stays valid after the game moves on — handy for comparing a
position before and after a sequence of actions.

In [3]:
report = GameReport(game)

print(report.summary())
print()
print(report.player_table())
print()
print(f"Bank           : {report.bank}")
print(f"Dev cards left : {report.development_cards_remaining}")
print(f"Robber at      : {report.robber_coordinate}")

Winner       : none (unfinished)
Turns        : 0
Actions      : 0
Longest Road : unclaimed
Largest Army : unclaimed (0 knights)

Color          VP   WOOD  BRICK  SHEEP  WHEAT    ORE   Road   Army
------------------------------------------------------------------
RED             0      0      0      0      0      0  False  False
WHITE           0      0      0      0      0      0  False  False
BLUE            0      0      0      0      0      0  False  False
ORANGE          0      0      0      0      0      0  False  False

Bank           : {'WOOD': 19, 'BRICK': 19, 'SHEEP': 19, 'WHEAT': 19, 'ORE': 19}
Dev cards left : 25
Robber at      : (-2, 1, 1)


## 4. Judging settlement spots

`BoardInspector` ranks nodes by expected per-roll production, which is the
first thing to look at when designing an opening-placement heuristic.

In [4]:
inspector = BoardInspector.from_game(game)

print("Land tiles (first 6):")
for tile in inspector.tile_summaries()[:6]:
    print(f"  {tile.coordinate}  {tile.resource:8s}  roll={tile.number}")

print()
print("Top-5 nodes by expected production:")
for node in inspector.top_producing_nodes(limit=5):
    print(f"  node {node.node_id:2d}  total={node.total:.3f}  {node.by_resource}")

Land tiles (first 6):
  (-2, 0, 2)  WHEAT     roll=10
  (-2, 1, 1)  DESERT    roll=None
  (-2, 2, 0)  SHEEP     roll=3
  (-1, -1, 2)  WOOD      roll=11
  (-1, 0, 1)  WHEAT     roll=4
  (-1, 1, 0)  ORE       roll=2

Top-5 nodes by expected production:
  node  2  total=0.417  {'BRICK': 0.1388888888888889, 'WOOD': 0.1388888888888889, 'WHEAT': 0.1388888888888889}
  node  9  total=0.361  {'WOOD': 0.1388888888888889, 'WHEAT': 0.1388888888888889, 'ORE': 0.08333333333333333}
  node  1  total=0.333  {'BRICK': 0.1388888888888889, 'WOOD': 0.1388888888888889, 'ORE': 0.05555555555555555}
  node  4  total=0.250  {'BRICK': 0.1388888888888889, 'WHEAT': 0.08333333333333333, 'ORE': 0.027777777777777776}
  node  8  total=0.250  {'WOOD': 0.1388888888888889, 'SHEEP': 0.027777777777777776, 'ORE': 0.08333333333333333}


## 5. Catanatron API scratch space

Direct pokes at the library, kept here on purpose: nothing reusable is defined,
these cells just document how `catanatron` advances a game.

| Method | Advances by |
|---|---|
| `game.play()` | Full game to completion |
| `game.play_tick()` | Exactly one player decision |
| `game.execute(action)` | One action you choose manually |

`Action` is a namedtuple `Action(color, action_type, value)`, and the legal
options are always on `state.playable_actions`.

In [5]:
duel = GameFactory([RandomPlayer(Color.RED), RandomPlayer(Color.BLUE)])
scratch = duel.create(seed=7)

print("Initial placement phase:")
while scratch.state.is_initial_build_phase:
    action = scratch.play_tick()
    print(f"  {action.color.value:6s}  {action.action_type.name:25s}  {action.value}")

print(f"\nBuild phase complete. Turns so far: {scratch.state.num_turns}")

Initial placement phase:
  BLUE    BUILD_SETTLEMENT           7
  BLUE    BUILD_ROAD                 (7, 8)
  RED     BUILD_SETTLEMENT           30
  RED     BUILD_ROAD                 (30, 31)
  RED     BUILD_SETTLEMENT           25
  RED     BUILD_ROAD                 (24, 25)
  BLUE    BUILD_SETTLEMENT           40
  BLUE    BUILD_ROAD                 (40, 44)

Build phase complete. Turns so far: 2


In [6]:
manual = duel.create(seed=99)

print("First 8 actions, chosen by hand:")
for _ in range(8):
    chosen = manual.state.playable_actions[0]
    logged = manual.execute(chosen)  # mutates in place, returns the resolved action
    print(f"  {logged.color.value:6s}  {logged.action_type.name:25s}  {logged.value}")

First 8 actions, chosen by hand:
  BLUE    BUILD_SETTLEMENT           0
  BLUE    BUILD_ROAD                 (0, 1)
  RED     BUILD_SETTLEMENT           2
  RED     BUILD_ROAD                 (2, 3)
  RED     BUILD_SETTLEMENT           4
  RED     BUILD_ROAD                 (3, 4)
  BLUE    BUILD_SETTLEMENT           6
  BLUE    BUILD_ROAD                 (6, 23)


In [7]:
# game.copy() forks a position — the basis for any lookahead search.
base = duel.create(seed=5)
for _ in range(50):
    if base.winning_color() is None:
        base.play_tick()

fork = base.copy()
fork.play()

print(f"Base still at turn {base.state.num_turns}")
print(f"Fork ended at turn {fork.state.num_turns}, winner: {fork.winning_color()}")

Base still at turn 18
Fork ended at turn 227, winner: Color.RED


## 6. Bots

`GreedySettlerBot` ([src/catan_bots/bots/greedy_settler.py](../src/catan_bots/bots/greedy_settler.py))
prioritises cities, then settlements, then dev cards, then roads, and breaks
ties with its own seeded RNG so runs are reproducible.

Built-in catanatron baselines to compare against: `RandomPlayer` (uniform),
`SimplePlayer` (always the first legal action), `WeightedRandomPlayer`
(weights cities over settlements over dev cards) and `VictoryPointPlayer`
(one-ply greedy on actual VP).

In [8]:
greedy_vs_random = GameFactory(
    [GreedySettlerBot(Color.RED, seed=1), RandomPlayer(Color.BLUE)]
)
played = greedy_vs_random.play(seed=42)

print(GameReport(played).summary())

Winner       : RED
Turns        : 124
Actions      : 337
Longest Road : RED
Largest Army : unclaimed (0 knights)


### Prototyping a new bot

Subclass `Player` and override `decide`. Prototype it here, and once it earns
its keep move it to `src/catan_bots/bots/` with a test — the same shape as
`GreedySettlerBot`.

In [9]:
class RoadRunnerBot(Player):
    """Prototype: chase Longest Road, ignore everything else."""

    def decide(self, game, playable_actions):
        roads = [a for a in playable_actions if a.action_type == ActionType.BUILD_ROAD]
        return roads[0] if roads else playable_actions[0]


prototype_arena = GameFactory([RoadRunnerBot(Color.RED), RandomPlayer(Color.BLUE)])
prototype_result = Tournament(prototype_arena, num_games=10, first_seed=0).run()

print(prototype_result.as_text())

Win rates over 10 games:
  BLUE        :  10/10  (100%)
  RED         :   0/10  (0%)
  draw        :   0/10  (0%)


## 7. Comparing bots over many games

`Tournament` plays a roster over consecutive seeds and tallies the winners, so
two runs of the same configuration give the same standings.

In [10]:
arena = GameFactory(
    [
        GreedySettlerBot(Color.RED, seed=1),
        WeightedRandomPlayer(Color.BLUE),
        VictoryPointPlayer(Color.ORANGE),
    ]
)
result = Tournament(arena, num_games=20, first_seed=0).run()

print(result.as_text())
print(f"\nMean turns per game: {result.mean_turns:.1f}")

Win rates over 20 games:
  RED         :   9/20  (45%)
  BLUE        :   7/20  (35%)
  ORANGE      :   4/20  (20%)
  draw        :   0/20  (0%)

Mean turns per game: 233.3


## 8. Collecting data during a game

`ActionTypeCounter` is a `GameAccumulator`: catanatron calls its hooks around
every action, so it collects data without touching game logic. Pass any number
of accumulators to `GameFactory.play`.

In [11]:
counter = ActionTypeCounter()
observed = arena.play(seed=7, accumulators=[counter])

print(f"Total actions: {counter.total_actions}")
for name, count in counter.ranked_counts():
    print(f"  {name:25s}: {count}")

Total actions: 550
  ROLL                     : 172
  END_TURN                 : 171
  MARITIME_TRADE           : 70
  MOVE_ROBBER              : 44
  BUILD_ROAD               : 37
  BUY_DEVELOPMENT_CARD     : 18
  BUILD_SETTLEMENT         : 12
  PLAY_KNIGHT_CARD         : 10
  DISCARD                  : 8
  BUILD_CITY               : 4
  PLAY_MONOPOLY            : 2
  PLAY_YEAR_OF_PLENTY      : 2


## 9. Reviewing the finished game

In [12]:
final = GameReport(observed)

print(final.summary())
print()
print(final.player_table())
print()
for snapshot in final.snapshots:
    print(
        f"  {snapshot.color.value:8s}"
        f"  settlements={snapshot.settlements}"
        f"  cities={snapshot.cities}"
        f"  roads={snapshot.roads}"
        f"  longest={snapshot.longest_road_length}"
        f"  knights={snapshot.knights_played}"
    )

print()
print("Last 10 actions:")
for line in final.recent_actions(limit=10):
    print(f"  {line}")

Winner       : BLUE
Turns        : 175
Actions      : 550
Longest Road : BLUE
Largest Army : RED (6 knights)

Color          VP   WOOD  BRICK  SHEEP  WHEAT    ORE   Road   Army
------------------------------------------------------------------
BLUE           10      2      0      2      3      2   True  False
RED             9      2      3      0      0      0  False   True
ORANGE          3      0      0      2      4      2  False  False

  BLUE      settlements=[]  cities=[7, 51, 53, 20]  roads=15  longest=9  knights=3
  RED       settlements=[11, 4, 14, 39, 2]  cities=[]  roads=15  longest=8  knights=6
  ORANGE    settlements=[34, 29, 27]  cities=[]  roads=7  longest=4  knights=1

Last 10 actions:
  RED       MARITIME_TRADE             ('WHEAT', 'WHEAT', 'WHEAT', None, 'ORE')
  RED       BUY_DEVELOPMENT_CARD       KNIGHT
  RED       PLAY_KNIGHT_CARD           None
  RED       MOVE_ROBBER                ((1, 0, -1), <Color.BLUE: 'BLUE'>, 'WHEAT')
  RED       MARITIME_TRADE         

## Where the code lives

| Concern | Module |
|---|---|
| Game construction, seeding, player resets | [src/catan_bots/games.py](../src/catan_bots/games.py) |
| Bots | [src/catan_bots/bots/](../src/catan_bots/bots/) |
| State and outcome reports | [src/catan_bots/analytics/reports.py](../src/catan_bots/analytics/reports.py) |
| Board and node production stats | [src/catan_bots/analytics/board_stats.py](../src/catan_bots/analytics/board_stats.py) |
| Tournaments | [src/catan_bots/analytics/tournament.py](../src/catan_bots/analytics/tournament.py) |
| Accumulators | [src/catan_bots/analytics/accumulators.py](../src/catan_bots/analytics/accumulators.py) |
| Tests | [tests/](../tests) |

## 10. Discarding on a seven

Catanatron does not model this decision: `discard_possibilities` returns one
placeholder action and the engine throws away a uniformly random half of the
hand. [src/catan_bots/rules/discard.py](../src/catan_bots/rules/discard.py)
replaces that with real policies, and
[src/catan_bots/runner.py](../src/catan_bots/runner.py) routes discard prompts
through them.

The reason the engine skips it is that enumerating *individual cards* explodes.
Enumerating resource *multisets* does not — same-resource cards are
interchangeable:

In [13]:
import math

from catanatron.models.enums import RESOURCES

from catan_bots.rules import enumerate_discards

for hand_size in (8, 10, 14, 20):
    # Spread the hand as evenly as possible across the five resources.
    hand = {r: hand_size // 5 for r in RESOURCES}
    for resource in RESOURCES[: hand_size % 5]:
        hand[resource] += 1
    k = hand_size // 2
    print(
        f"hand {hand_size:2d}, discard {k:2d}: "
        f"{len(enumerate_discards(hand, k)):4d} multiset options "
        f"vs {math.comb(hand_size, k):,} per-card combinations"
    )

hand  8, discard  4:   26 multiset options vs 70 per-card combinations
hand 10, discard  5:   51 multiset options vs 252 per-card combinations
hand 14, discard  7:  124 multiset options vs 3,432 per-card combinations
hand 20, discard 10:  381 multiset options vs 184,756 per-card combinations


### What the policies choose

`BuildPlanDiscard` keeps the cards that fund the best reachable build, then
spends what is left on partial progress, then on the resources this player
produces least often. `UniformRandomDiscard` reproduces the engine's behaviour
from a seeded RNG, and is the ablation baseline.

In [14]:
from catanatron import Color
from catanatron.models.player import RandomPlayer

from catan_bots import GameFactory
from catan_bots.rules import BuildPlanDiscard, UniformRandomDiscard, production_rate
from catan_bots.rules.discard import hand_counts

seats = [RandomPlayer(c) for c in (Color.RED, Color.BLUE, Color.ORANGE, Color.WHITE)]
position = GameFactory(seats).create(seed=42)
while position.state.is_initial_build_phase:
    position.play_tick()

key = f"P{position.state.color_to_index[Color.RED]}"
for resource, count in {"WHEAT": 2, "ORE": 3, "WOOD": 4, "SHEEP": 1}.items():
    position.state.player_state[f"{key}_{resource}_IN_HAND"] = count

hand = hand_counts(position.state, Color.RED)
print("hand:", {r: n for r, n in hand.items() if n})
print(
    "own production per roll:",
    {r: round(production_rate(position.state, Color.RED, r), 3) for r in hand},
)
print()
print("build-plan discards:", BuildPlanDiscard().resolve(position, Color.RED))
print("random discards    :", UniformRandomDiscard(seed=3).resolve(position, Color.RED))

hand: {'WOOD': 4, 'SHEEP': 1, 'WHEAT': 2, 'ORE': 3}
own production per roll: {'WOOD': 0.056, 'BRICK': 0.0, 'SHEEP': 0.0, 'WHEAT': 0.167, 'ORE': 0.028}

build-plan discards: ('WOOD', 'WOOD', 'WOOD', 'WOOD', 'SHEEP')
random discards    : ('WOOD', 'ORE', 'WOOD', 'ORE', 'SHEEP')


### Comparing rules in one game

`seat_registry` gives named seats their own rule, so the same bot can play
itself with only the discard rule differing.

In [15]:
from catan_bots.runner import seat_registry

registry = seat_registry(
    {Color.RED: BuildPlanDiscard(), Color.ORANGE: BuildPlanDiscard()},
    default=UniformRandomDiscard(seed=0),
)
compared = GameFactory(seats, discard_policy=registry).play(seed=9)
discards = [a for a in compared.state.actions if a.action_type.name == "DISCARD"]

print(f"discard events: {len(discards)}")
for action in discards[:6]:
    print(f"  {action.color.value:7s} discarded {action.value}")

discard events: 23
  RED     discarded ('BRICK', 'SHEEP', 'SHEEP', 'SHEEP')
  ORANGE  discarded ('SHEEP', 'WHEAT', 'WHEAT', 'WHEAT')
  RED     discarded ('BRICK', 'SHEEP', 'SHEEP', 'WHEAT', 'WHEAT')
  RED     discarded ('BRICK', 'SHEEP', 'ORE', 'ORE')
  BLUE    discarded ('ORE', 'ORE', 'SHEEP', 'WOOD')
  RED     discarded ('BRICK', 'SHEEP', 'WHEAT', 'WHEAT', 'WHEAT')
